In [ ]:
# %pip install feedparser beautifulsoup4

In [ ]:
import feedparser
import requests
import json
import time
import numpy as np

from scipy import spatial
from sklearn.manifold import TSNE
import plotly.express as px

# 1. Get the feed

In [ ]:
feed_URL="https://vickiboykis.com/index.xml"

d = feedparser.parse(feed_URL)

d['feed']['title']

# 2. Get the posts

In [ ]:
# d.keys()
posts = d['entries']
# for k in posts[0].keys():
#     print(f"[{k}]\n{posts[0][k]}\n\n\n")

# we only have the latest 10 posts because RSS
# we could think about using waybackpack to get older ones
# (see e.g. https://stackoverflow.com/questions/576552/how-do-i-fetch-all-old-items-on-an-rss-feed)
# AND if we want to allow for this, provide either longer feed or a way to paginate in the past
print(len(posts))

# shall we get the full HTML post and parse it with bs4?
# print(posts[0]['content'][0]['value'])
# from bs4 import BeautifulSoup 
# soup = BeautifulSoup(posts[0]['content'][0]['value'])

# not all RSS feeds have full posts, plus full text has to
# be parsed AND split into 256-token-long chunks, so let'
# just consider summaries (they will be cut to the first
# 256 tokens automatically by the model, see
# https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2#intended-uses

# First 1024 chars (not really 256 tokens but just wanted to
# get a rough idea of how much we can see in the first chunk
# of the summary
print(posts[0]['summary'][:1024])

summaries = [post['summary'] for post in posts]
titles = [post['title'] for post in posts]

# 3. Calculate embeddings

In [ ]:
def get_embedding(text):

    try:
        response = requests.request(
            url = "http://localhost:8080/embedding", 
            method = "POST",
            data = {"content": text},
        )
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"Request failed: {e}")
        raise

    return json.loads(response.text)["embedding"]

tt = time.time()
embeddings = []
i = 0
for t in summaries:
    i+=1
    embeddings.append(get_embedding(t))
    if not i%10:
        print(".", end="")

embeddings = np.array(embeddings)
print(time.time()-tt)
embeddings.shape

# 4. Use KDTree to calculate K-nearest neighbors of a given status

In [ ]:
tree = spatial.KDTree(embeddings)

sentence = "large language models topic detection"

idx = tree.query(get_embedding(sentence), k=5)[1]
for i in idx:
    print(f"{i}:{titles[i]}\n{summaries[i]}\n\n")

# 5. Use TSNE to plot the statuses in a 2D space

In [ ]:
# note you can play with the perplexity parameter to have more or less crisp clusters
# (smaller values of perplexity tends to have tighter, more sparse clusters, while 
# larger values return larger, more globular and possibly overlapping ones)
tsne = TSNE(n_components=2, 
            random_state=42,
            perplexity=1.5
)
projections = tsne.fit_transform(embeddings)

fig = px.scatter(
    projections, x=0, y=1,
    hover_name = titles,
    # color=lbls,
)
fig.show()


# 6. Rinse and repeat with another blog, another field ('tags')

In [ ]:
feed_URL="https://pluralistic.net/feed/"

d = feedparser.parse(feed_URL)

all_tags = []
for entry in d['entries']:
    tags = ", ".join([tag['term'] for tag in entry['tags'] if tag['term']!="Uncategorized"])
    all_tags.append(tags)

In [ ]:
tt = time.time()
embeddings = []
i = 0
for t in all_tags:
    i+=1
    embeddings.append(get_embedding(t))
    if not i%10:
        print(".", end="")

embeddings = np.array(embeddings)
print(time.time()-tt)

In [ ]:
from scipy import spatial
tree = spatial.KDTree(embeddings)

sentence = "crypto mining sustainability"
# sentence = "drm, enshittification, history, microsoft, microsoft research, mp3s, ngscb, podcasts, retrospectives, trusted computing"

idx = tree.query(get_embedding(sentence), k=5)[1]
for i in idx:
    print(f"{i}:{all_tags[i]}\n\n")

# NOTES:

* post summaries work well as long as:
  * they are not too long
  * they actually summarize the post

* full posts do not work well unless they are first split into chunks / summarized

* publishers should make access to older content easy (i.e. provide all content, or allow pagination) to allow for sync and local search
